# IDS 2025/2026 End-to-End Notebook

Centralized learning + Federated Learning (IID, Non-IID, SMPC, DP, HE, Hybrid).

## 1) Setup

In [ ]:
!pip install -q -r ../requirements.txt

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt

from src.data import load_ids_dataset, partition_iid, partition_non_iid, client_class_distribution
from src.centralized import run_centralized_training
from src.federated import run_full_experiment_suite

## 2) Configure dataset path

In [ ]:
DATA_PATH = '/path/to/ids_trend_2025_2026.csv'  # <-- change this
TARGET_COLUMN = 'label'
MODEL_NAME = 'resmlp'
DEVICE = 'cuda'

## 3) Load and inspect dataset

In [ ]:
data = load_ids_dataset(DATA_PATH, target_column=TARGET_COLUMN)
print('feature_dim:', data.feature_dim)
print('num_classes:', data.num_classes)
print('train/val/test:', len(data.X_train), len(data.X_val), len(data.X_test))

## 4) Visualize IID vs Non-IID client distribution

In [ ]:
iid_parts = partition_iid(data.y_train, num_clients=10)
non_iid_parts = partition_non_iid(data.y_train, num_clients=10, dirichlet_alpha=0.3)

iid_df = client_class_distribution(data.y_train, iid_parts)
non_iid_df = client_class_distribution(data.y_train, non_iid_parts)

display(iid_df.head())
display(non_iid_df.head())

## 5) Centralized baseline

In [ ]:
centralized = run_centralized_training(
    data=data,
    model_name=MODEL_NAME,
    epochs=20,
    batch_size=1024,
    lr=1e-3,
    weight_decay=1e-4,
    device=DEVICE,
)
centralized

## 6) Federated suite (IID / Non-IID / SMPC / DP / HE / Hybrid)

In [ ]:
federated = run_full_experiment_suite(
    data=data,
    rounds=15,
    num_clients=10,
    local_epochs=2,
    model_name=MODEL_NAME,
    device=DEVICE,
)
list(federated.keys())

## 7) Compare final metrics

In [ ]:
rows = []
rows.append({
    'experiment': 'centralized',
    'test_acc': centralized['test_acc'][-1],
    'test_f1': centralized['test_f1'][-1],
})
for name, result in federated.items():
    rows.append({
        'experiment': name,
        'test_acc': result['test_acc'][-1],
        'test_f1': result['test_f1'][-1],
    })

summary_df = pd.DataFrame(rows).sort_values('test_f1', ascending=False)
display(summary_df)

## 8) Save results

In [ ]:
results = {'centralized': centralized, 'federated': federated}
with open('ids_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Saved ids_results.json')